### Ken Vellian
#### CSC 555: Mining Big Data
#### Final Project
#### Due: Nov 19, 2024 | Professor: Ahmed Abid

#### The purpose of this notebook is to use PySpark to explore and preprocess the data so we can stage it in S3 for Athena to call SQL queries against the data.

In [1]:
import boto3
import time
import pandas as pd

In [2]:
import os
# AWS configuration
AWS_REGION = 'us-east-1'  # Your AWS region
S3_BUCKET_NAME = 'finalprojectbucket-kenvellian'  # Replace with your actual S3 bucket name
S3_KEY = 'data/'  # Folder where your CSV data is stored
TABLE_NAME = 'card_transdata_table'  # Athena table name

# AWS credentials (Replace with your live credentials)
AWS_ACCESS_KEY = os.environ["AWS_ACCESS_KEY_ID"]

AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]

AWS_SESSION_TOKEN = os.environ["AWS_SESSION_TOKEN"]



In [3]:
# Initialize Athena client with the credentials
athena_client = boto3.client(
    'athena',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

In [4]:
# Function to run Athena query
def run_athena_query(query, database, output_location):
    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': output_location}
    )
    return response['QueryExecutionId']

# Function to check query status
def check_query_status(query_execution_id):
    while True:
        response = athena_client.get_query_execution(QueryExecutionId=query_execution_id)
        status = response['QueryExecution']['Status']['State']
        
        # Check if the query has finished successfully
        if status == 'SUCCEEDED':
            print("Query succeeded!")
            break
        elif status in ['FAILED', 'CANCELLED']:
            print(f"Query {status.lower()}.")
            raise Exception(f"Query failed or was cancelled: {response}")
        
        # Wait for a short time before checking the status again
        time.sleep(2)
        
# Function to get query results and convert them to a pandas DataFrame
def get_query_results_as_dataframe(query_execution_id):
    results_paginator = athena_client.get_paginator('get_query_results')
    results_iter = results_paginator.paginate(QueryExecutionId=query_execution_id)

    columns = []
    rows = []

    for results_page in results_iter:
        if not columns:
            columns = [col['Label'] for col in results_page['ResultSet']['ResultSetMetadata']['ColumnInfo']]
        
        for row in results_page['ResultSet']['Rows'][1:]:  # Skip header row
            rows.append([col.get('VarCharValue', None) for col in row['Data']])

    df = pd.DataFrame(rows, columns=columns)
    return df

In [5]:
# Define the SQL query to create the Athena table (assuming CSV format)
create_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS `{TABLE_NAME}` (
    distance_from_home DOUBLE,
    distance_from_last_transaction DOUBLE,
    ratio_to_median_purchase_price DOUBLE,
    repeat_retailer INT,
    used_chip INT,
    used_pin_number INT,
    online_order INT,
    fraud INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{S3_BUCKET_NAME}/{S3_KEY}'
"""

# Print the query to ensure it's correct
print(create_table_query.strip())

# Execute the query to create the Athena table
response = athena_client.start_query_execution(
    QueryString=create_table_query.strip(),
    QueryExecutionContext={'Database': 'default'},  # Modify if you have a custom database
    ResultConfiguration={'OutputLocation': f's3://{S3_BUCKET_NAME}/query_results/'}
)

# Output the status
print("Athena table creation query submitted.")

CREATE EXTERNAL TABLE IF NOT EXISTS `card_transdata_table` (
    distance_from_home DOUBLE,
    distance_from_last_transaction DOUBLE,
    ratio_to_median_purchase_price DOUBLE,
    repeat_retailer INT,
    used_chip INT,
    used_pin_number INT,
    online_order INT,
    fraud INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://finalprojectbucket-kenvellian/data/'
Athena table creation query submitted.


In [6]:
# Run the Athena query
query = f'SELECT fraud, COUNT(*) as fraud_count FROM "{TABLE_NAME}" GROUP BY fraud;'
query_execution_id = run_athena_query(query, 'default', f's3://{S3_BUCKET_NAME}/query_results/')

# Check the query status
check_query_status(query_execution_id)

# Fetch the query results into a pandas DataFrame
df = get_query_results_as_dataframe(query_execution_id)

# Display the results
df

Query succeeded!


,fraud,fraud_count
0,1,87403
1,None,1
2,0,912597
